# 02 — LLM feature discovery and extraction with `llm-feature-gen`

**Goal.** Show the two-step pipeline the project already uses, and make it reproducible.

The library works in two stages:

1. **Discovery** — show the model a pile of transcripts *without labels* and ask it to propose
   a feature schema (feature name, description, possible values). Written to a JSON file.
2. **Generation** — for every document, ask the model to assign a value to each feature in
   that schema. Written to one CSV per class.

**Which data goes where.** Discovery runs on `overview/` (= `train.extra`), which has no
labels at all. That is deliberate: the schema must be designed without ever seeing the class
structure. Generation then runs on `train/`. The test folder is not used.

> **Cost note.** A full extraction is one LLM call per document. This notebook therefore
> defaults to `RUN_LLM = False` and loads the feature table that was already produced
> (`OutputsQwen/`). Set `RUN_LLM = True` when you have the endpoint available and want to
> regenerate. The code that runs is the same either way.

In [2]:
!pip install llm-feature-gen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 472.5 kB/s eta 0:00:00


In [11]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from llm_feature_gen.providers.local_provider import LocalProvider

In [12]:
DATA.mkdir(parents=True, exist_ok=True)
(DATA / "overview").mkdir(parents=True, exist_ok=True)
print(f"Created directories: {DATA} and {DATA / 'overview'}")

Created directories: fileDataset and fileDataset/overview


## The provider

`LocalProvider` is the right class for an OpenAI-compatible endpoint such as the school's
Ollama server. (`OpenAIProvider` does not accept a custom `base_url` on its non-Azure path,
so it will not work against `llm.vse.cz`.)

In [13]:
RUN_LLM = True

BASE_URL = "https://llm.vse.cz/ollama/v1/"
API_KEY = "ollama"
MODEL = "qwen3.5:122b"

if RUN_LLM:
    provider = StreamingLocalProvider(
        base_url=BASE_URL,
        api_key=API_KEY,
        default_text_model=MODEL,
        temperature=0.0,
        max_tokens=8000,
    )

    print("Provider ready:", provider.text_model)
else:
    provider = None

Provider ready: qwen3.5:122b


## Step 1 — Discovery prompt

The library ships a default discovery prompt, but it is written for a *two-hidden-class*
setting and says so explicitly. That is a problem here: telling the model there are two groups
invites it to write feature descriptions of the form *"one group does X, the other does Y"*,
which is exactly what happened in the first run of this project. The schema then encodes the
class structure before any labels are used.

The prompt below is label-blind: it never mentions groups, classes, diagnoses or health.

In [14]:
V2_PROMPT = """\
You are annotating a transcript of spontaneous Czech speech. A person was shown a drawing of a
lakeshore scene and asked to describe it aloud. The text is an automatic transcription.

Your job is to COUNT observable linguistic events and QUOTE the evidence for each count.
You are an annotator, not an evaluator.

RULES
- Quote evidence verbatim from the transcript, in Czech. Never translate or paraphrase.
- If a category has no instances, return an empty list. Empty is a valid answer.
- Never infer anything about the speaker: not health, ability, age, education or mood.
- Do not compare this speaker to anyone else or to any norm.
- Count occurrences, not impressions. Two hedges in one sentence are two entries.
- Return only JSON.

CATEGORIES
1. named_entities - every distinct object, creature or person named. Lemmatise; list each once.
2. specific_action_verbs - verbs naming a particular manner of action.
3. generic_verbs - verbs of bare existence, possession, location or unspecified movement.
4. complete_propositions - integer. Clauses with explicit subject AND predicate AND one more argument.
5. locative_expressions - phrases placing something somewhere.
6. regions_referenced - which of "water", "land", "sky" are explicitly referred to.
7. hedge_spans - expressions of uncertainty about what is depicted.
8. deictic_spans - places where the speaker points instead of naming.
9. metacomment_spans - remarks about the speaker's own describing or remembering, or about the task.
10. repeated_content_lemmas - content words used more than once, with counts.
11. self_corrections - integer. Restarts, replacements or retractions.
12. diminutive_or_affective_forms - noun forms marked as diminutive or affectionate.
13. quantity_expressions - numerals or quantifiers applied to things in the scene.

Return exactly this JSON and nothing else:
{"named_entities": [], "specific_action_verbs": [], "generic_verbs": [],
 "complete_propositions": 0, "locative_expressions": [], "regions_referenced": [],
 "hedge_spans": [], "deictic_spans": [], "metacomment_spans": [],
 "repeated_content_lemmas": [{"lemma": "", "count": 0}], "self_corrections": 0,
 "diminutive_or_affective_forms": [], "quantity_expressions": []}
"""

for w in ["group", "class", "diagnos", "impair", "patient", "healthy"]:
    assert w not in V2_PROMPT.lower(), f"Prompt leaks label information: {w}"

print("V2 prompt is label-blind — OK")

V2 prompt is label-blind — OK


In [15]:
import json
from pathlib import Path
import pandas as pd

from llm_feature_gen.providers.local_provider import LocalProvider

DATA = Path("fileDataset")

if not (DATA / "train").exists() and (DATA / "fileDataset" / "train").exists():
    DATA = DATA / "fileDataset"

if not DATA.exists() and Path("train").exists():
    DATA = Path(".")

OUT = Path("outputs_v2")
OUT.mkdir(exist_ok=True)

print("Dataset:", DATA.resolve())
print("Output:", OUT.resolve())

Dataset: /content/fileDataset
Output: /content/outputs_v2


## Step 1 (run) — discover a schema from the unlabelled pool

In [16]:
import json as _json
import time as _time
import openai as _openai

from openai import BadRequestError as _BadRequestError
from llm_feature_gen.providers.local_provider import LocalProvider


class StreamingLocalProvider(LocalProvider):
    """
    LocalProvider variant using streaming responses.
    Streaming keeps the HTTP connection active while Qwen is generating.
    """

    def _chat_json(
        self,
        deployment_name,
        system_prompt,
        user_content,
        json_mode=False
    ):
        if json_mode and "JSON" not in system_prompt:
            system_prompt += "\nRespond in strict JSON format."

        kwargs = {}

        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        backoff = 2

        for attempt in range(self.max_retries):
            try:
                chunks = []

                with self.client.chat.completions.create(
                    model=deployment_name,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_content},
                    ],
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                    stream=True,
                    **kwargs,
                ) as stream:

                    for chunk in stream:
                        delta = chunk.choices[0].delta.content

                        if delta:
                            chunks.append(delta)

                text = "".join(chunks)

                try:
                    return _json.loads(text)

                except Exception:
                    extracted = self._extract_json(text)

                    if extracted:
                        if isinstance(extracted, list):
                            return {"features": extracted}
                        return extracted

                    if json_mode:
                        raise ValueError(
                            f"Invalid JSON response:\n{text}"
                        )

                    return {"features": text}

            except _BadRequestError as e:

                if json_mode and "json_object" in str(e):
                    json_mode = False
                    kwargs.pop("response_format", None)
                    continue

                raise

            except _openai.RateLimitError:

                if attempt < self.max_retries - 1:
                    _time.sleep(backoff)
                    backoff *= 2
                    continue

                raise

        raise RuntimeError(
            "Unable to obtain response after maximum retries."
        )


BASE_URL = "https://llm.vse.cz/ollama/v1/"
API_KEY = "ollama"
MODEL = "qwen3.5:122b"

provider = StreamingLocalProvider(
    base_url=BASE_URL,
    api_key=API_KEY,
    default_text_model=MODEL,
    temperature=0.0,
    max_tokens=8000,
)

print("Provider ready")
print("Model:", provider.text_model)
print("Base URL:", provider.base_url)

Provider ready
Model: qwen3.5:122b
Base URL: https://llm.vse.cz/ollama/v1/


In [17]:
train_files = sorted((DATA / "train").glob("*.txt"))

print("Training transcripts:", len(train_files))

texts = [
    p.read_text(encoding="utf-8").strip()
    for p in train_files
]

print("Example transcript:")
print(texts)

Training transcripts: 0
Example transcript:
[]


In [18]:
if RUN_LLM:
    results = provider.text_features(
        text_list=texts,
        prompt=V2_PROMPT,
        feature_gen=True,
    )

    print("Generated features:", len(results))

Generated features: 0


## Step 2 — Generate feature values for the training documents

`generate_features_from_texts` expects a root folder containing one subfolder per class. Our
`train/` folder already has `negative/` and `positive/`, so it works directly. `Class` in the
output CSV is just the folder name — it is a bookkeeping column, not something the model saw.

In [ ]:
from llm_feature_gen.generate import generate_features_from_texts

if RUN_LLM:
    paths = generate_features_from_texts(
        root_folder=str(DATA / "train"),
        discovered_features_path=str(SCHEMA_PATH),
        output_dir=str(OUT / "train_features_v1"),
        classes=["negative", "positive"],
        merge_to_single_csv=True,
        merged_csv_name="train_all_feature_values.csv",
        provider=provider,
    )
    print(paths)
else:
    print("skipped — using OutputsQwen/train_all_feature_values.csv")

skipped — using OutputsQwen/train_all_feature_values.csv


## Load the feature table

Either the one we just produced, or the existing one from `OutputsQwen/`.

In [ ]:
CANDIDATES = [
    OUT / "train_features_v1" / "train_all_feature_values.csv",
    Path("OutputsQwen") / "train_all_feature_values.csv",
    Path("train_all_feature_values.csv"),
]
feat_path = next((p for p in CANDIDATES if p.exists()), None)
if feat_path is None:
    raise FileNotFoundError(f"no feature CSV found; looked in {[str(p) for p in CANDIDATES]}")

features = pd.read_csv(feat_path)
FEATURE_COLS = [c for c in features.columns if c not in ("File", "Class", "raw_llm_output")]

print("loaded:", feat_path)
print(f"{len(features)} rows x {len(FEATURE_COLS)} features")
print("classes:", features.Class.value_counts().to_dict())
print("\nfeatures:")
for c in FEATURE_COLS:
    print("  -", c)

loaded: OutputsQwen/train_all_feature_values.csv
241 rows x 10 features
classes: {'negative': 171, 'positive': 70}

features:
  - narrative_coherence
  - uncertainty_markers
  - action_verb_tense_consistency
  - entity_specificity_level
  - self_referential_metacommentary
  - spatial_organization_pattern
  - emotional_interpretation_inference
  - lexical_variation_and_repetition
  - quantification_precision
  - discourse_marker_usage


## What one row looks like

Each feature is a short categorical label. `raw_llm_output` keeps the model's original JSON,
which is useful when a value looks wrong and you want to see what was actually returned.

In [ ]:
row = features.iloc[0]
print("file:", row.File, "| class:", row.Class, "\n")
for c in FEATURE_COLS:
    print(f"  {c:38} {row[c]}")

file: 100tr0.txt | class: negative 

  narrative_coherence                    simple_sentences
  uncertainty_markers                    occasional
  action_verb_tense_consistency          strict_present
  entity_specificity_level               basic_attributes
  self_referential_metacommentary        minimal
  spatial_organization_pattern           random_jump
  emotional_interpretation_inference     purely_physical
  lexical_variation_and_repetition       moderate_variation
  quantification_precision               estimates_only
  discourse_marker_usage                 sparse


## Coverage check — did every document get every feature?

In [ ]:
missing = features[FEATURE_COLS].isna().sum()
print("missing values per feature:")
print(missing.to_string() if missing.sum() else "  none — every document has every feature")

levels = {c: sorted(features[c].dropna().unique()) for c in FEATURE_COLS}
print("\ndistinct values actually used:")
for c, v in levels.items():
    print(f"  {c:38} {len(v)}  {v}")

missing values per feature:
  none — every document has every feature

distinct values actually used:
  narrative_coherence                    4  ['connected_narrative', 'fragmented_list', 'simple_sentences', 'telegraphic_utterances']
  uncertainty_markers                    4  ['frequent', 'none', 'occasional', 'rare']
  action_verb_tense_consistency          4  ['inconsistent_aspect', 'mixed_tenses', 'not observable', 'strict_present']
  entity_specificity_level               3  ['basic_attributes', 'descriptive_details', 'generic_only']
  self_referential_metacommentary        5  ['absent', 'dominant', 'frequent', 'minimal', 'moderate']
  spatial_organization_pattern           4  ['foreground_first', 'left_to_right', 'random_jump', 'systematic_scan']
  emotional_interpretation_inference     4  ['high_inference', 'minimal_inference', 'moderate_inference', 'purely_physical']
  lexical_variation_and_repetition       4  ['high_repetition', 'high_variation', 'low_variation', 'moderate_va

In [ ]:
features.to_csv(OUT / "features_v1_train.csv", index=False)
json.dump({"feature_columns": FEATURE_COLS}, open(OUT / "feature_columns.json", "w"), indent=2)
print(f"wrote outputs/features_v1_train.csv")

wrote outputs/features_v1_train.csv


### Summary

* Discovery runs on the unlabelled `overview/` pool; generation runs on `train/`.
* The discovery prompt was rewritten to be label-blind. The original library default tells the
  model there are two hidden classes, and the first version of this project's schema contained
  descriptions of the form *"one group tends to… while the other…"*. That is worth mentioning
  to your supervisor — it is a methodological fix, not a cosmetic one.
* The result is 10 categorical features over 241 training documents.
* Whether those features are any good is notebook 03.